# Symptom-Based Disease Diagnosis System
### Healthcare Dataset | Multi-Class Classification

Predict the most likely disease a patient has based on their reported symptoms,
supporting faster and more accurate preliminary diagnosis.

**Dataset:** Training.csv & Testing.csv
**Features:** 132 binary symptom columns (0 = absent, 1 = present)
**Target:** prognosis (42 disease classes)
**Approach:** CRISP-DM Methodology

## 0. Business Understanding

### Background
Misdiagnosis and delayed diagnosis are major problems in healthcare globally.
In many settings, especially in areas with limited specialist access, patients
wait days or weeks before receiving a diagnosis. A symptom-based ML system can
provide an instant preliminary diagnosis to support clinical decision-making.

### Business Problem
A healthcare provider wants to build a system that, given a patient's reported
symptoms, returns the most likely disease and its probability. The system should:
- Help general practitioners narrow down differential diagnoses quickly
- Prioritise patients for specialist referrals based on predicted disease severity
- Serve as a first-line triage tool in resource-limited clinics

### ML Objective
Build a **multi-class classification model** that maps 132 binary symptom inputs
to one of 42 disease classes (the `prognosis` target).

### Success Criteria
- **Accuracy** is the primary metric — each disease has a unique, non-overlapping
  symptom profile, so high accuracy is achievable and expected
- Model must correctly classify all 42 diseases in the test set
- **predict_proba()** used to return top-5 disease probabilities for clinical use

In [1]:
# Import core libraries for data manipulation, visualisation and numerical operations
# warnings.simplefilter('ignore') suppresses non-critical warnings for cleaner output
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.simplefilter("ignore")

## 1. Data Loading
Loading separate Training and Testing CSV files.
Both files have the same 132 symptom features and 1 target column (prognosis).

In [9]:
# Load separate training and testing CSV files provided with the dataset
# Print shapes to confirm both files loaded correctly
import pandas as pd

train_df = pd.read_csv("Training.csv")
test_df = pd.read_csv("Testing.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (4920, 134)
Test shape: (42, 133)


## 2. Data Understanding
Exploring structure, column names, data types and a preview of the training data.

In [11]:
# Inspect structure: column names, data types, non-null counts and a sample of rows
print(train_df.info())
print(train_df.head())
print(train_df.columns)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4920 entries, 0 to 4919
Columns: 134 entries, itching to Unnamed: 133
dtypes: float64(1), int64(132), object(1)
memory usage: 5.0+ MB
None
   itching  skin_rash  nodal_skin_eruptions  continuous_sneezing  shivering  \
0        1          1                     1                    0          0   
1        0          1                     1                    0          0   
2        1          0                     1                    0          0   
3        1          1                     0                    0          0   
4        1          1                     1                    0          0   

   chills  joint_pain  stomach_pain  acidity  ulcers_on_tongue  ...  scurring  \
0       0           0             0        0                 0  ...         0   
1       0           0             0        0                 0  ...         0   
2       0           0             0        0                 0  ...         0   
3       0   

### 2.1 Class Distribution
Checking how many samples exist for each disease (prognosis).
A balanced distribution means no resampling techniques are needed.

In [13]:
# Check how many samples exist per disease — equal counts confirm a balanced dataset
print(train_df['prognosis'].value_counts())

prognosis
Fungal infection                           120
Allergy                                    120
GERD                                       120
Chronic cholestasis                        120
Drug Reaction                              120
Peptic ulcer diseae                        120
AIDS                                       120
Diabetes                                   120
Gastroenteritis                            120
Bronchial Asthma                           120
Hypertension                               120
Migraine                                   120
Cervical spondylosis                       120
Paralysis (brain hemorrhage)               120
Jaundice                                   120
Malaria                                    120
Chicken pox                                120
Dengue                                     120
Typhoid                                    120
hepatitis A                                120
Hepatitis B                                120
Hep

## 3. Data Cleaning

### 3.1 Drop Unnamed Column
The dataset contains an extra empty column 'Unnamed: 133' — likely a CSV export artifact.
Dropping it from both train and test datasets using errors='ignore' as a safety measure.

In [15]:
# Drop the extra empty column created as a CSV export artifact
# errors='ignore' prevents errors if the column is already absent
train_df = train_df.drop(columns=['Unnamed: 133'], errors='ignore')
test_df = test_df.drop(columns=['Unnamed: 133'], errors='ignore')

### 3.2 Null Values
Checking for missing values in both train and test datasets.
All symptom columns are binary (0/1) so no imputation should be needed.

In [19]:
# Confirm zero missing values across all symptom columns in both train and test sets
print(train_df.isnull().sum().sum())
print(test_df.isnull().sum().sum())

0
0


### 3.3 Duplicate Values
Checking for repeated rows in the training dataset.

In [21]:
# Check for repeated rows — duplicates would bias model training
print("Duplicate rows in train:", train_df.duplicated().sum())

Duplicate rows in train: 4616


### 3.4 Remove Duplicates
Dropping duplicate rows from training data and confirming the new shape.

In [ ]:
# Remove duplicate rows and confirm the updated shape of the training data
train_df = train_df.drop_duplicates()
print("Train shape after removing duplicates:", train_df.shape)

## 4. Feature Engineering
Splitting both train and test datasets into features (X) and target (y).
No encoding needed — all symptom columns are already binary (0/1).
No scaling needed — all features are on the same scale already.
No train/test split needed — separate files already provided.

In [29]:
# Split features (X) and target (y) from both train and test datasets
# No encoding or scaling needed — all symptom columns are already binary (0/1)
X_train = train_df.drop(columns=['prognosis'])
y_train = train_df['prognosis']

X_test = test_df.drop(columns=['prognosis'])
y_test = test_df['prognosis']

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(304, 132) (304,)
(42, 132) (42,)


## 5. Modelling
Importing all required classifiers and evaluation metrics.
3 models will be trained: Logistic Regression, KNN, and SVM.

In [31]:
# Import all classifiers and evaluation utilities needed for modelling
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

### 5.1 Logistic Regression
Baseline linear classifier for multiclass disease prediction.
max_iter=1000 set to ensure convergence on the 132-feature dataset.
Includes full evaluation: confusion matrix and classification report.

In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score

# Load data
train_df = pd.read_csv("Training.csv")
test_df = pd.read_csv("Testing.csv")

# Drop unwanted column
train_df = train_df.drop(columns=['Unnamed: 133'], errors='ignore')
test_df = test_df.drop(columns=['Unnamed: 133'], errors='ignore')

# Features and target
X_train = train_df.drop(columns=['prognosis'])
y_train = train_df['prognosis']

X_test = test_df.drop(columns=['prognosis'])
y_test = test_df['prognosis']

# Model
log_model = LogisticRegression(random_state=42, max_iter=1000)

# Train model
log_model.fit(X_train, y_train)

# Predict and evaluate on train data
ypred_train = log_model.predict(X_train)
print("Train Accuracy:", accuracy_score(y_train, ypred_train))

# Cross validation on training data
cv_score = cross_val_score(log_model, X_train, y_train, cv=5, scoring="accuracy").mean()
print("CV Score:", cv_score)

# Predict and evaluate on test data
ypred_test = log_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, ypred_test))

# Extra evaluation
print("\nConfusion Matrix:\n", confusion_matrix(y_test, ypred_test))
print("\nClassification Report:\n", classification_report(y_test, ypred_test))

Train Accuracy: 1.0
CV Score: 1.0
Test Accuracy: 1.0

Confusion Matrix:
 [[1 0 0 ... 0 0 0]
 [0 1 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 0 1 0]
 [0 0 0 ... 0 0 1]]

Classification Report:
                                          precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00         1
                                   AIDS       1.00      1.00      1.00         1
                                   Acne       1.00      1.00      1.00         1
                    Alcoholic hepatitis       1.00      1.00      1.00         1
                                Allergy       1.00      1.00      1.00         1
                              Arthritis       1.00      1.00      1.00         1
                       Bronchial Asthma       1.00      1.00      1.00         1
                   Cervical spondylosis       1.00      1.00      1.00         1
                            Chicken pox       1.00  

### 5.2 K-Nearest Neighbors (KNN)
Distance based classifier. GridSearchCV tunes n_neighbors (1-50)
and distance metric p (1=Manhattan, 2=Euclidean).
Best estimator is selected and evaluated on train and test sets.

In [39]:
# Hyperparameter Tuning
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score

estimator = KNeighborsClassifier()
param_grid = {"n_neighbors": list(range(1, 50)), "p": [1, 2]}

knn_grid = GridSearchCV(estimator, param_grid, scoring="accuracy", cv=5)
knn_grid.fit(X_train, y_train)

# KNN with best Hyperparameters
knn_grid.best_estimator_

# Modelling
knn_model = knn_grid.best_estimator_
knn_model.fit(X_train, y_train)

# Predict and evaluate on Train Data
ypred_train = knn_model.predict(X_train)
print("Train Accuracy:", accuracy_score(y_train, ypred_train))

# Cross Validation Score
print("CV score:", cross_val_score(knn_model, X_train, y_train, cv=5, scoring="accuracy").mean())

# Predict and evaluate on Test Data
ypred_test = knn_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, ypred_test))

Train Accuracy: 1.0
CV score: 1.0
Test Accuracy: 1.0


### 5.3 Support Vector Machine (SVM)
Finds the optimal hyperplane to separate disease classes.
GridSearchCV tunes C (regularization strength) and kernel type.
Works well with high-dimensional binary feature spaces like this symptom dataset.

In [43]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score

estimator = SVC(random_state=42)
param_grid = {"C": [0.01, 0.1, 1], "kernel": ["linear", "rbf", "sigmoid", "poly"]}

svm_grid = GridSearchCV(estimator, param_grid, scoring="accuracy", cv=5)
svm_grid.fit(X_train, y_train)

# SVM with best Hyperparameters
svm_grid.best_estimator_

# Modelling
svm_model = svm_grid.best_estimator_
svm_model.fit(X_train, y_train)

# Predict and evaluate on Train Data
ypred_train = svm_model.predict(X_train)
print("Train Accuracy:", accuracy_score(y_train, ypred_train))

# Cross Validation on Train Data
print("CV Score:", cross_val_score(svm_model, X_train, y_train, cv=5, scoring="accuracy").mean())

# Predict and evaluate on Test Data
ypred_test = svm_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, ypred_test))

Train Accuracy: 1.0
CV Score: 1.0
Test Accuracy: 1.0


## 6. Prediction on New Patient
Creating a new patient record with specific symptoms set to 1.
All other symptoms default to 0 (not present).
Using Logistic Regression to predict the most likely disease
and showing top 5 diseases ranked by probability.

In [45]:
import pandas as pd

# Create a blank patient with all symptoms = 0
new_patient = pd.DataFrame(0, index=[0], columns=X_train.columns)

# Set symptoms present in the patient
new_patient.loc[0, 'itching'] = 1
new_patient.loc[0, 'skin_rash'] = 1
new_patient.loc[0, 'nodal_skin_eruptions'] = 1
new_patient.loc[0, 'dischromic _patches'] = 1

# Predict disease
predicted_disease = log_model.predict(new_patient)[0]
print("Predicted Disease:", predicted_disease)

# Predict probabilities
predicted_probabilities = log_model.predict_proba(new_patient)[0]

prob_df = pd.DataFrame({
    'Disease': log_model.classes_,
    'Probability': predicted_probabilities
}).sort_values(by='Probability', ascending=False)

print("\nTop 5 Predicted Diseases:")
print(prob_df.head(5))

Predicted Disease: Fungal infection

Top 5 Predicted Diseases:
                Disease  Probability
15     Fungal infection     0.992543
14        Drug Reaction     0.001547
2                  Acne     0.000733
27             Impetigo     0.000452
9   Chronic cholestasis     0.000387


## 7. Model Selection Note

All 3 models — Logistic Regression, KNN, and SVM — achieved **100% accuracy**
on both training and test sets.

This is expected because:
- The dataset has **132 binary symptom features**
- Each disease maps to a **unique and specific symptom combination**
- There is **no overlap** between disease patterns
- The data is clean, balanced and perfectly structured

Running additional models (Random Forest, XGBoost etc.) would not
add any value since the problem is already solved with perfect accuracy.